# 経済データ分析Ⅱ　第7回
# 2値の被説明変数と線形確率モデル

ここまでの回では、消費や床面積のように、被説明変数がいろいろな値をとる場合を扱った。今回は、被説明変数が「持ち家に住んでいれば1、そうでなければ0」のように、0か1の2つの値だけをとる場合を考える。第6回で説明変数に使ったダミー変数を、今回は被説明変数に使う。

このような被説明変数を OLS でそのまま回帰したものを**線形確率モデル**という。係数の読み方と、線形確率モデルに残る問題を調べる。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中のセルから実行して「変数が見つからない」というエラー（`NameError`）が出たら、先頭のセルから実行し直してください。

JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。データは Notebook の中で作るので、CSV ファイルの準備は要りません。

## 1. 準備

次のセルで、今回使うライブラリを読み込む。`japanize_matplotlib_jlite` は、図の文字を日本語で表示するためのパッケージで、経済データ分析Ⅰでも使った。ブラウザでは `piplite.install()` でこのパッケージを入れる。`if sys.platform == 'emscripten':` から始まる部分は、ブラウザ（JupyterLite）で実行しているときに必要なライブラリを用意する処理である。`smf` は第6回と同じく、回帰式を文字で書いて推定する機能である。`from scipy import stats` は、確率分布の値を計算する機能を読み込む。

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels', 'scipy'])
    import piplite
    await piplite.install('japanize-matplotlib-jlite')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite
import statsmodels.formula.api as smf
from scipy import stats

## 2. 持ち家かどうかのデータ

### 2.1. 年齢と年収

総務省統計局『令和5年住宅・土地統計調査』の[第45-1表](https://www.e-stat.go.jp/stat-search/files?stat_infid=000040209924)には、家計を主に支える人が雇用者の世帯（会社などに雇われて働く人の世帯）の持ち家率がある。年齢でみると、30〜34歳の33.2%から60〜64歳の79.4%まで、年齢が高いほど高い。年間収入でみても、300〜400万円の51.4%に対して、1,000〜1,500万円では79.9%である。

この傾向をもとに、500世帯の人工データを作る。実際の世帯の記録ではない。

まず、世帯主の年齢を作る。`np.random.default_rng(202611)` は第3回と同じく、乱数を作る仕組みを用意して `rng` という名前を付ける。シードの202611を変えなければ、何度実行しても同じデータになる。`rng.integers(25, 65, size=n)` は、25以上65未満の整数を同じ確率で選ぶので、年齢は25〜64歳になる。

In [ ]:
rng = np.random.default_rng(202611)
n = 500
age = rng.integers(25, 65, size=n)
print(f'年齢：{age.min()}〜{age.max()}歳')

次に、世帯の年収を作る。年収は百万円単位で、7百万円に、第3回と同じく対数正規分布の倍率を掛けて世帯ごとにばらつかせる。`mean=-0.08` と `sigma=0.4` は、倍率の平均が1になるように選んだ値である。`np.median()` は中央値を求める。

In [ ]:
income = 7 * rng.lognormal(mean=-0.08, sigma=0.4, size=n)
print(f'年収の中央値：{np.median(income):.2f} 百万円')

年収の中央値は6.42百万円（642万円）である。

### 2.2. 標準正規分布の累積分布関数

続いて、世帯ごとの持ち家の確率を年齢と年収から決める。年齢と年収の1次式は、年齢や年収の値によって0未満にも1超にもなるが、確率は0と1の間の値である。1次式の値を0と1の間に収めるために、標準正規分布の累積分布関数を使う。

**標準正規分布**は、平均0・標準偏差1の正規分布である。第3回の仮定6で説明したように、正規分布は平均を中心に左右対称な釣鐘型の分布で、この釣鐘の形を表す関数を**密度関数**という。密度関数の曲線が高いところほど、その近くの値が出やすい。値がある範囲に入る確率は、その範囲で曲線の下にある部分の面積に等しく、曲線の下の面積は全体で1になる。

標準正規分布に従う値が $z$ 以下になる確率を $\Phi(z)$ と書く。$\Phi(z)$ は、密度関数の曲線の下で $z$ より左にある部分の面積である。左側から $z$ まで面積を積み重ねた値なので、$\Phi$ を標準正規分布の**累積分布関数**という。

`stats.norm.cdf()` は $\Phi(z)$ を計算する。$z=0$ と $z=1$ での値を求める。

In [ ]:
print(f'Φ(0)：{stats.norm.cdf(0):.3f}')
print(f'Φ(1)：{stats.norm.cdf(1):.3f}')

$\Phi(0)=0.5$ である。密度関数の曲線は0を中心に左右対称なので、0より左の面積は全体の半分になる。$\Phi(1)=0.841$ は、標準正規分布に従う値が84.1%の確率で1以下になることを表す。

$\Phi(1)$ にあたる面積を図で見る。まず、横軸の点を用意する。`np.linspace(-4, 4, 200)` は第6回と同じく、−4から4までに200個の点を等しい間隔で並べる。

In [ ]:
z = np.linspace(-4, 4, 200)
print(f'最小：{z.min():.0f}、最大：{z.max():.0f}、点の数：{len(z)}')

`stats.norm.pdf()` は、それぞれの点で密度関数の高さを計算する。`.max()` で最も高い値を調べる。

In [ ]:
density = stats.norm.pdf(z)
print(f'密度関数の最も高い値：{density.max():.3f}')

密度関数は $z=0$ の近くで最も高く、高さは0.399である。

密度関数の曲線を描き、$z=1$ より左の部分を塗る。

- `z_left` は、塗る範囲の−4から1までに並べた100個の点である。
- `ax.fill_between(横軸の点, 曲線の高さ)` は、曲線と横軸の間を塗る。`z_left` を渡すので、$z=1$ より左だけが塗られる。
- `alpha=0.3` は色を薄くし、`label` は凡例に出す名前を決める。
- `ax.legend(loc='upper left')` は凡例を左上に置く。

In [ ]:
z_left = np.linspace(-4, 1, 100)
fig, ax = plt.subplots()
ax.plot(z, density)
ax.fill_between(z_left, stats.norm.pdf(z_left), alpha=0.3, label='1以下になる確率 Φ(1)')
ax.set(xlabel='z', ylabel='密度関数の高さ')
ax.legend(loc='upper left')
plt.show()

塗った部分の面積が $\Phi(1)=0.841$ で、右側に残る部分の面積は $1-0.841=0.159$ である。

$z$ を大きくすると、塗る部分が右へ広がるので、面積 $\Phi(z)$ は大きくなる。$z$ を横軸に、$\Phi(z)$ を縦軸にとって、累積分布関数を描く。

- `stats.norm.cdf(z)` は、200個の点それぞれで $\Phi(z)$ を計算する。
- `ax.scatter(1, stats.norm.cdf(1), color='red')` は、$z=1$ の点を赤で描く。

In [ ]:
fig, ax = plt.subplots()
ax.plot(z, stats.norm.cdf(z))
ax.scatter(1, stats.norm.cdf(1), color='red')
ax.set(xlabel='z', ylabel='Φ(z)')
plt.show()

赤い点の高さが、前の図で塗った面積の0.841である。曲線は、$z$ が小さいところでは0に近く、$z$ が大きくなるにつれて1に近づく。$\Phi(z)$ は確率なので、どんな $z$ を入れても0と1の間に入る。そのため、年齢と年収の1次式がどんな値になっても、それを $\Phi$ に入れた値は持ち家の確率として使える。

### 2.3. 持ち家の確率と持ち家の有無

世帯ごとの持ち家の確率 $p_i$ を、年齢と年収の1次式を $\Phi$ に入れて決める。

$$p_i=\Phi(-2.3+0.04\times年齢_i+0.09\times年収_i)$$

係数は、2.1節で挙げた持ち家率に近くなるように決めた。`p[:5]` は、`p` の先頭から5つの値を取り出す。

In [ ]:
p = stats.norm.cdf(-2.3 + 0.04 * age + 0.09 * income)
print(p[:5].round(3))

最初の5世帯の確率は0.467、0.428、0.410、0.475、0.599である。

この確率で、持ち家（1）かそうでないか（0）を決める。`rng.random(n)` は0以上1未満の数を同じ確率で選ぶ。その数が $p_i$ より小さければ1とすると、確率 $p_i$ で1になる。`.astype(int)` は、比較の結果（`True` と `False`）を1と0に変える。

In [ ]:
owned = (rng.random(n) < p).astype(int)
print(owned[:5])

最初の5世帯は、0、0、0、1、0になった。

4つの変数を、`pd.DataFrame()` で1つの表にまとめる。

In [ ]:
df = pd.DataFrame({'age': age, 'income': income, 'p': p, 'owned': owned})
df.head().round(3)

`age` は世帯主の年齢、`income` は年収（百万円）、`p` は持ち家の真の確率、`owned` は持ち家なら1、そうでなければ0である。5行目の世帯は、持ち家の確率が0.599と5世帯の中で最も高いが、`owned` は0になった。確率が高くても、持ち家にならない世帯はある。実際のデータでは `owned` だけが観測でき、`p` は分からない。

### 2.4. 0と1の被説明変数の平均

$y_i$ が0か1だけをとるとき、その期待値は1になる確率に等しい。

$$E(y\mid x)=0\times P(y=0\mid x)+1\times P(y=1\mid x)=P(y=1\mid x)$$

データでは、`owned` の平均が持ち家の世帯の割合になる。

In [ ]:
print(f"owned の平均：{df['owned'].mean():.3f}")

500世帯の52.0%が持ち家である。

年齢を4つの区分に分けて、区分ごとに `owned` の平均と真の確率 `p` の平均を並べる。`pd.cut()` は第6回と同じく、数値を区分に分ける。

In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[24, 34, 44, 54, 64],
                         labels=['25〜34歳', '35〜44歳', '45〜54歳', '55〜64歳'])
df[['age', 'age_group']].head()

`.groupby('age_group', observed=True)` は第6回と同じく、区分ごとに行をまとめる。`[['owned', 'p']].mean()` で、2つの列の平均を区分ごとに求める。

In [ ]:
df.groupby('age_group', observed=True)[['owned', 'p']].mean().round(3)

区分ごとの `owned` の平均（持ち家の割合）は、25〜34歳で0.313、55〜64歳で0.762で、真の確率の平均0.322、0.751に近い。0と1の被説明変数の回帰は、この確率を説明変数で表すことにあたる。

## 3. 線形確率モデル

### 3.1. 推定

持ち家かどうかを、年齢と年収に OLS で回帰する。

$$P(y=1\mid x)=E(y\mid x)=\beta_0+\beta_1\,年齢+\beta_2\,年収$$

確率を説明変数の1次式で表すので、線形確率モデルという。推定の仕方は第6回までの回帰と同じである。

In [ ]:
lpm = smf.ols('owned ~ age + income', data=df).fit()
print(lpm.params.round(4))

年齢の係数0.0156は、年収が同じなら、世帯主の年齢が1歳高いと持ち家の確率が約1.6ポイント（0.0156）高いことを表す。年収の係数0.0299は、年齢が同じなら、年収が1百万円多いと持ち家の確率が約3.0ポイント高いことを表す。係数がそのまま確率の変化として読めることが、線形確率モデルの長所である。

### 3.2. 誤差の分散と頑健な標準誤差

線形確率モデルの誤差は $u=y-p$ である。$y=1$（確率 $p$）なら $u=1-p$、$y=0$（確率 $1-p$）なら $u=-p$ になるので、誤差の分散は

$$\mathrm{Var}(u\mid x)=p(1-p)^2+(1-p)p^2=p(1-p)$$

である。$p=0.5$ なら0.25、$p=0.9$ なら0.09で、確率 $p$ が説明変数によって変わると、誤差の分散も変わる。線形確率モデルでは誤差が必ず不均一分散になるので、第5回の頑健な標準誤差を使う。第5回と同じく `cov_type='HC0'` で推定し、通常の標準誤差と並べる。

In [ ]:
lpm_hc0 = smf.ols('owned ~ age + income', data=df).fit(cov_type='HC0')
pd.DataFrame({'推定値': lpm.params, '通常の標準誤差': lpm.bse,
              '頑健な標準誤差': lpm_hc0.bse}).round(4)

推定値は同じで、標準誤差だけが変わる。このデータでは2つの標準誤差の差は小さく、年収では0.0075と0.0072である。それでも、線形確率モデルでは誤差の分散が一定という仮定が成り立たないので、頑健な標準誤差を使う。

### 3.3. 予測確率

線形確率モデルの当てはめ値は、持ち家の予測確率と読める。新しい世帯の予測確率は、`lpm.predict()` に年齢と年収の表を渡して計算する。25歳で年収1百万円の世帯と、64歳で年収20百万円の世帯を予測する。

In [ ]:
new = pd.DataFrame({'age': [25, 64], 'income': [1, 20]})
lpm.predict(new).round(3)

1つ目の世帯の予測確率は0.032、2つ目の世帯は1.209である。確率は1を超えないので、2つ目の値は確率として意味をもたない。

予測確率は、推定した係数を1次式に入れた値である。2つ目の世帯について、式で計算して確かめる。

$$\hat{p}=\hat\beta_0+\hat\beta_1\times64+\hat\beta_2\times20$$

係数は `lpm.params['age']` のように名前で取り出せる。切片の名前は `'Intercept'` である。

In [ ]:
b = lpm.params
print(f"式で計算した予測確率：{b['Intercept'] + b['age'] * 64 + b['income'] * 20:.3f}")

式で計算しても1.209になり、`lpm.predict()` の値と同じである。予測確率は説明変数の1次式なので、年齢や年収が大きくなるといくらでも大きくなる。

真の確率と線形確率モデルの予測確率を、年収を中央値の6.42百万円に固めて、年齢の関数として図に描く。まず、横軸の年齢を用意する。`np.linspace(25, 64, 100)` は2.2節と同じく、25から64までに100個の点を等しい間隔で並べる。

In [ ]:
age_grid = np.linspace(25, 64, 100)
mid = np.median(income)
print(f'最初の点：{age_grid[0]:.0f}歳、最後の点：{age_grid[-1]:.0f}歳、固める年収：{mid:.2f} 百万円')

それぞれの年齢で、真の確率と線形確率モデルの予測確率を計算する。

In [ ]:
true_line = stats.norm.cdf(-2.3 + 0.04 * age_grid + 0.09 * mid)
lpm_line = b['Intercept'] + b['age'] * age_grid + b['income'] * mid
print(f'25歳：真の確率 {true_line[0]:.3f}、線形確率モデル {lpm_line[0]:.3f}')

25歳では、真の確率が0.235、線形確率モデルが0.194である。2本の線と、各世帯の `owned`（0か1）の点を描く。`alpha=0.3` で点の色を薄くして、重なりを見やすくする。凡例は2.2節と同じく左上に置く。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df['age'], df['owned'], s=10, alpha=0.3)
ax.plot(age_grid, true_line, label='真の確率')
ax.plot(age_grid, lpm_line, label='線形確率モデル')
ax.set(xlabel='世帯主の年齢（歳）', ylabel='持ち家（1）かどうか')
ax.legend(loc='upper left')
plt.show()

被説明変数は0か1なので、点は上下2本の線に並ぶ。真の確率はゆるやかな曲線で、線形確率モデルはそれを直線で近似している。年齢の範囲の中では2本はよく似ている。直線は、年齢の範囲の外まで延ばすと、0未満や1超の値になる。

データの500世帯の当てはめ値（`lpm.fittedvalues`）の最小と最大を調べる。

In [ ]:
fitted = lpm.fittedvalues
print(f'最小：{fitted.min():.3f}、最大：{fitted.max():.3f}')

当てはめ値は0.090から1.110までである。0未満か1超の世帯を数える。`|` は「または」を表し、`.sum()` で `True` の数を数える。

In [ ]:
print(f'0未満か1超の世帯の数：{((fitted < 0) | (fitted > 1)).sum()}')

1を超えた世帯が4つある。データの範囲の中でも、確率の上限を超える予測が出ている。

確率がいつも0〜1に入るように、2.2節の $\Phi$ のような曲線で確率を表すのがプロビットとロジットで、第8回で扱う。

## 参考文献

この回の構成は下の本の17.1.1節（Linear Probability Models）に沿っている。この本は著者のサイトで無料で読める。

- [Heiss, F. and Brunner, D. (2024) Using Python for Introductory Econometrics, 2nd edition](https://www.urfie.net/downloads/PDF/UPfIE_web.pdf)

## 関数の資料

- [NumPy：乱数の生成（Generator）](https://numpy.org/doc/stable/reference/random/generator.html)
- [SciPy：標準正規分布（`stats.norm`）](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
- [Matplotlib：曲線の下を塗る（`fill_between`）](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html)
- [statsmodels：結果の予測（`predict`）](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.predict.html)